In [7]:
import spacy
import json
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
from nltk.stem import PorterStemmer
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pandas as pd
import random
import pickle
from utils import load_data, create_embedding_matrix, preprocess_text, to_padding, pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]



In [ ]:
class MyTokenizer:
    def __init__(self):
        self.word_index = {"<PAD>": 0, "<UNK>": 1}
        self.idx_to_token = {0: "<PAD>", 1: "<UNK>"}
        
    def fit_on_texts(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word_index:
                    self.word_index[word] = len(self.word_index)
                    self.idx_to_token[self.word_index[word]] = word

    def text_to_sequences(self, text):
        return [self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()]
    
    def texts_to_sequences(self, texts):
        return [[self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()] for text in texts]


    def encode(self, text, max_length):
        tokens = self.text_to_sequences(text)
        if len(tokens) < max_length:
            tokens += [self.word_index["<PAD>"]] * (max_length - len(tokens))
        else:
            tokens = tokens[:max_length]
        return tokens
    
    def __call__(self, claims, evidences, max_length=512, return_tensors="pt"):
        self.fit_on_texts([claims, evidences])
        encoded_claims = self.encode(claims, max_length)
        encoded_evidences = self.encode(evidences, max_length)
        if return_tensors == "pt":
            return {
                "input_ids": torch.tensor([encoded_claims, encoded_evidences], dtype=torch.long)
            }
        return {"input_ids": [encoded_claims, encoded_evidences]}


In [4]:
nlp = spacy.load("en_core_web_lg")  # Load NER model
# nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer =  SnowballStemmer('english')

train_claims_data = load_data('../data/train-claims.json')
evidence_data = load_data('../data/evidence.json')
dev_claims_data = load_data('../data/dev-claims.json')
# evidence_map = load_data('../data/curated/preprocessed_evidence_map.json')  
# evidence_map = load_data('../data/curated/mild_nostopwords_evidence.json')  
# filtered_evidence_map = load_data('../data/curated/mild_nostopwords_filtered_evidence.json')

In [5]:
# data_for_dataframe = []
# for eid, evidence_text in evidence_data.items():
#     data_for_dataframe.append({
# 			'eid': eid,
#             'evidence_entities': extract_entities(evidence_text)
#         })
    
# # Create DataFrame
# evidence_entities_df = pd.DataFrame(data_for_dataframe)
# evidence_entities_df

,eid,evidence_entities
0,evidence-0,"[(John Bennet Lawes, PERSON), (English, NORP)]"
1,evidence-1,"[(Lindberg, PERSON), (the age of 16, DATE), (N..."
2,evidence-2,"[(``Boston (Ladies of Cambridge, ORG), (Vampir..."
3,evidence-3,"[(Gerald Francis Goyer, PERSON), (October 20, ..."
4,evidence-4,"[(ECT, ORG)]"
...,...,...
1208822,evidence-1208822,[]
1208823,evidence-1208823,"[(Fyrde, PERSON)]"
1208824,evidence-1208824,"[(Dragon Storm, PERSON)]"
1208825,evidence-1208825,"[(Zeriuani, PERSON), (Slavs, NORP), (Zeriuani,..."


In [8]:
# evidence_entities_df.to_csv("evidence_entities.csv", index=False)
evidence_entities_df = pd.read_csv("evidence_entities.csv")

In [9]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim_text': claim_text,
            'claim_entities': extract_entities(claim_text),
            'evidence': eids
        })
    
claim_entities_df = pd.DataFrame(data_for_dataframe)
claim_entities_df

,claim_id,claim_text,claim_entities,evidence
0,claim-752,[South Australia] has the most expensive elect...,"[(South Australia, LOC)]","[evidence-67732, evidence-572512]"
1,claim-375,when 3 per cent of total annual global emissio...,"[(3 per cent, MONEY), (Australia, GPE), (1.3 p...","[evidence-996421, evidence-1080858, evidence-2..."
2,claim-1266,This means that the world is now 1C warmer tha...,"[(1C, CARDINAL)]","[evidence-889933, evidence-694262]"
3,claim-871,"“As it happens, Zika may also be a good model ...","[(Zika, PERSON), (second, ORDINAL)]","[evidence-422399, evidence-702226, evidence-28..."
4,claim-2164,Greenland has only lost a tiny fraction of its...,"[(Greenland, GPE)]","[evidence-52981, evidence-264761, evidence-947..."
...,...,...,...,...
149,claim-2400,"'To suddenly label CO2 as a ""pollutant"" is a d...","[(Earth, LOC)]","[evidence-409365, evidence-127519, evidence-85..."
150,claim-204,"after a natural orbitally driven warming, atmo...","[(800 years later, DATE)]","[evidence-368192, evidence-261690, evidence-20..."
151,claim-1426,Many of the world’s coral reefs are already ba...,[],"[evidence-1124018, evidence-995813, evidence-1..."
152,claim-698,A recent study led by Lawrence Livermore Natio...,"[(Lawrence Livermore, PERSON), (National Labor...",[evidence-660755]


In [11]:
def match_evidence(claim_entities, evidence_entities_df, k=3):
    matched_evidences = []
    
    for _, row in evidence_entities_df.iterrows():
        evidence_id = row['eid']
        entities = row['evidence_entities']
        intersection = set(claim_entities) & set(entities)
        matched_evidences.append((evidence_id, len(intersection)))

    # Sort the matched evidences by the length of the intersection (descending order)
    matched_evidences.sort(key=lambda x: x[1], reverse=True)

    # Return the top 3 evidence IDs with the most intersections
    top_evidence_ids = [evidence_id for evidence_id, _ in matched_evidences[:k]]
    return top_evidence_ids

claim_entities_df['top_evidence_ids'] = claim_entities_df['claim_entities'].apply(lambda x: match_evidence(x, evidence_entities_df))


In [12]:
claim_entities_df

,claim_id,claim_text,claim_entities,evidence,top_evidence_ids
0,claim-752,[South Australia] has the most expensive elect...,"[(South Australia, LOC)]","[evidence-67732, evidence-572512]","[evidence-56198, evidence-94222, evidence-161322]"
1,claim-375,when 3 per cent of total annual global emissio...,"[(3 per cent, MONEY), (Australia, GPE), (1.3 p...","[evidence-996421, evidence-1080858, evidence-2...","[evidence-444, evidence-483, evidence-596]"
2,claim-1266,This means that the world is now 1C warmer tha...,"[(1C, CARDINAL)]","[evidence-889933, evidence-694262]","[evidence-416116, evidence-671846, evidence-79..."
3,claim-871,"“As it happens, Zika may also be a good model ...","[(Zika, PERSON), (second, ORDINAL)]","[evidence-422399, evidence-702226, evidence-28...","[evidence-185, evidence-394, evidence-487]"
4,claim-2164,Greenland has only lost a tiny fraction of its...,"[(Greenland, GPE)]","[evidence-52981, evidence-264761, evidence-947...","[evidence-662, evidence-5928, evidence-6525]"
...,...,...,...,...,...
149,claim-2400,"'To suddenly label CO2 as a ""pollutant"" is a d...","[(Earth, LOC)]","[evidence-409365, evidence-127519, evidence-85...","[evidence-215, evidence-636, evidence-681]"
150,claim-204,"after a natural orbitally driven warming, atmo...","[(800 years later, DATE)]","[evidence-368192, evidence-261690, evidence-20...","[evidence-0, evidence-1, evidence-2]"
151,claim-1426,Many of the world’s coral reefs are already ba...,[],"[evidence-1124018, evidence-995813, evidence-1...","[evidence-0, evidence-1, evidence-2]"
152,claim-698,A recent study led by Lawrence Livermore Natio...,"[(Lawrence Livermore, PERSON), (National Labor...",[evidence-660755],"[evidence-624586, evidence-109404, evidence-58..."


In [15]:
claim_entities_df_output = claim_entities_df.set_index("claim_id")
claim_entities_df_output = claim_entities_df_output.drop(columns=['claim_entities', 'evidence'])
claim_entities_df_output = claim_entities_df_output.rename(columns={'top_evidence_ids': 'evidence'})
claim_entities_df_output

,claim_text,evidence
claim_id,,
claim-752,[South Australia] has the most expensive elect...,"[evidence-56198, evidence-94222, evidence-161322]"
claim-375,when 3 per cent of total annual global emissio...,"[evidence-444, evidence-483, evidence-596]"
claim-1266,This means that the world is now 1C warmer tha...,"[evidence-416116, evidence-671846, evidence-79..."
claim-871,"“As it happens, Zika may also be a good model ...","[evidence-185, evidence-394, evidence-487]"
claim-2164,Greenland has only lost a tiny fraction of its...,"[evidence-662, evidence-5928, evidence-6525]"
...,...,...
claim-2400,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-215, evidence-636, evidence-681]"
claim-204,"after a natural orbitally driven warming, atmo...","[evidence-0, evidence-1, evidence-2]"
claim-1426,Many of the world’s coral reefs are already ba...,"[evidence-0, evidence-1, evidence-2]"


In [16]:
claim_entities_df_output.to_json("ner_predicted.json", orient="index")

In [ ]:
def match_evidence(claim_entities, evidence_entities_df, k=3):
    matched_evidences = []
    
    for _, row in evidence_entities_df.iterrows():
        evidence_id = row['eid']
        entities = row['evidence_entities']
        intersection = set(claim_entities) & set(entities)
        matched_evidences.append((evidence_id, len(intersection)))

    # Sort the matched evidences by the length of the intersection (descending order)
    matched_evidences.sort(key=lambda x: x[1], reverse=True)

    # Return the top 3 evidence IDs with the most intersections
    top_evidence_ids = [evidence_id for evidence_id, _ in matched_evidences[:k]]
    return top_evidence_ids

claim_entities_df['top_evidence_ids'] = claim_entities_df['claim_entities'].apply(lambda x: match_evidence(x, evidence_entities_df))
